In [1]:
# %% [markdown]
# # Hierarchical MNL with Mixture-of-Normals on Margarine Data
# This notebook reads the JSON data exported from the `bayesm` R package and 
# fits the HMNL model using `liesel` and `jax`. The prior parameters strictly 
# match the original R configuration.

# %%
import json
import time
import datetime
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from matplotlib.lines import Line2D

import liesel.model as lsl
import liesel.goose as gs
from tensorflow_probability.substrates.jax.experimental import distributions as tfde
from tensorflow_probability.python.internal.backend.jax.compat import v2 as tf
import tensorflow_probability.substrates.jax.distributions as tfd
import tensorflow_probability.substrates.jax.bijectors as tfb

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (16, 9)

In [2]:
# %%
import json
import numpy as np

data_path = "margarine_data.json"
print(f"Loading data from {data_path}...")

with open(data_path, 'r') as f:
    lgtdata_list = json.load(f)

n_params = 10
n_alts = 10
K_components = 5 # As specified in the R script

X_list, y_list, unit_idx_list, Z_list = [], [], [], []

for i, hh_data in enumerate(lgtdata_list):
    # USE np.atleast_1d TO PREVENT SCALAR ERRORS ON n_obs=1
    y_i = np.atleast_1d(hh_data['y']) - 1
    
    # R exported row-major (n_obs * n_alts, n_params)
    X_i_flat = np.array(hh_data['X'])
    Z_raw = hh_data['Z_raw']
    
    n_obs = len(y_i)
    
    # Reshape to (n_obs, n_alts, n_params)
    X_i = X_i_flat.reshape((n_obs, n_alts, n_params))
    
    for t in range(n_obs):
        X_list.append(X_i[t])
        y_list.append(y_i[t])
        unit_idx_list.append(i)
        
    Z_list.append(Z_raw)

# Process Demographics Matrix Z
Z = np.array(Z_list, dtype=float)

# Z_raw is [Income, Fam_Size]. We take log(Income) as done in your R processing
# Masking out missing/zero/negative incomes if they exist
valid_inc = Z[:, 0] > 0
Z[valid_inc, 0] = np.log(Z[valid_inc, 0])
Z[~valid_inc, 0] = np.nan

# Mean-center Z
Z_mean = np.nanmean(Z, axis=0)
Z = Z - Z_mean

# Replace NaNs with 0 (which is the mean since it's centered)
Z = np.nan_to_num(Z)

# rhierMnlRwMixture adds an intercept column to Z implicitly, we add it explicitly.
import jax.numpy as jnp
Z_with_int = np.column_stack((np.ones(len(Z)), Z))

choice_data = {
    "X": jnp.array(X_list),
    "y": jnp.array(y_list),
    "Z": jnp.array(Z_with_int),
    "unit_idx": jnp.array(unit_idx_list),
    "n_units": len(lgtdata_list),
    "n_params": n_params,
    "n_demos": Z_with_int.shape[1],  # Intercept + 2 demographics = 3
    "K": K_components,
    "n_alts": n_alts
}

print(f"Prepared Data: {choice_data['n_units']} households, {len(y_list)} total observations.")
print(f"X shape: {choice_data['X'].shape}, y shape: {choice_data['y'].shape}, Z shape: {choice_data['Z'].shape}")

Loading data from margarine_data.json...
Prepared Data: 516 households, 4470 total observations.
X shape: (4470, 10, 10), y shape: (4470,), Z shape: (516, 3)


In [3]:
# %% [markdown]
# ## 2. Model Definition
# Adapted to enforce `A_delta = 1/16` (`0.0625`) to match the `A_mu = 1/16` specification 
# mapped to prior `A` on $\Delta$ from your original R code snippet.

# %%
def make_wishart(df, scale_tril):
    return tfd.WishartTriL(
        df=df,
        scale_tril=scale_tril,
        input_output_cholesky=True,
        validate_args=False
    )

def make_mvn_precision(loc, precision_factor):
    return tfde.MultivariateNormalPrecisionFactorLinearOperator(
        loc=loc,
        precision_factor=tf.linalg.LinearOperatorLowerTriangular(precision_factor),
        validate_args=False
    )

def build_mixture_hmnl_model(
        data_dict,
        A_delta=0.0625,    # 1/16: prior precision on Delta; matches `A = A_mu * diag` in R
        a_mu=0.01,         # Prior precision scalar for mu_k|Sigma_k (bayesm defaults to 0.01)
        dirichlet_a=5.0    # Dirichlet concentration; bayesm defaults to 5
):
    n_params = int(data_dict["n_params"])
    n_units  = int(data_dict["n_units"])
    K        = int(data_dict["K"])
    has_Z    = data_dict.get("Z") is not None

    nu    = float(n_params + 3)
    V     = nu * jnp.eye(n_params)
    V_inv = jnp.linalg.inv(V)
    Vinv_chol   = jnp.linalg.cholesky(V_inv)
    Vinv_chol_K = jnp.broadcast_to(Vinv_chol[None], (K, n_params, n_params))

    alpha = jnp.ones(K) * dirichlet_a
    pvec = lsl.Var.new_param(
        value=jnp.ones(K) / K,
        distribution=lsl.Dist(tfd.Dirichlet, concentration=alpha),
        name="pvec"
    )
    pvec_latent = pvec.transform(tfb.SoftmaxCentered(), name="pvec_latent")

    sigma_inv_chol_k = lsl.Var.new_param(
        value=jnp.broadcast_to(jnp.eye(n_params)[None], (K, n_params, n_params)),
        distribution=lsl.Dist(
            make_wishart,
            df=jnp.full(K, nu),
            scale_tril=Vinv_chol_K
        ),
        name="sigma_inv_chol_k"
    )
    sigma_inv_chol_k_latent = sigma_inv_chol_k.transform(
        tfb.FillScaleTriL(), name="sigma_inv_chol_k_latent"
    )

    mu_prec_factor_k = lsl.Var.new_calc(
        lambda L: jnp.sqrt(a_mu) * L,
        L=sigma_inv_chol_k,
        name="mu_prec_factor_k"
    )

    mu_k = lsl.Var.new_param(
        value=jnp.zeros((K, n_params)),
        distribution=lsl.Dist(
            make_mvn_precision,
            loc=jnp.zeros(n_params),
            precision_factor=mu_prec_factor_k
        ),
        name="mu_k"
    )

    if has_Z:
        n_demos = data_dict["Z"].shape[1]
        Z_var = lsl.Var.new_obs(data_dict["Z"], name="Z_obs")
        Delta_prec_factor = jnp.sqrt(A_delta) * jnp.eye(n_params)

        Delta = lsl.Var.new_param(
            value=jnp.zeros((n_demos, n_params)),
            distribution=lsl.Dist(
                make_mvn_precision,
                loc=jnp.zeros(n_params),
                precision_factor=Delta_prec_factor
            ),
            name="Delta"
        )
        z_delta = lsl.Var.new_calc(
            lambda z, d: z @ d, z=Z_var, d=Delta, name="z_delta"
        )

    sigma_chol_k = lsl.Var.new_calc(
        lambda L: jax.vmap(
            lambda Lk: jnp.linalg.cholesky(
                jnp.linalg.inv(Lk @ Lk.T) + 1e-6 * jnp.eye(n_params)
            )
        )(L),
        L=sigma_inv_chol_k,
        name="sigma_chol_k"
    )

    if has_Z:
        beta_loc = lsl.Var.new_calc(
            lambda zd, mu: zd[:, None, :] + mu[None, :, :],
            zd=z_delta, mu=mu_k,
            name="beta_loc"
        )
    else:
        beta_loc = lsl.Var.new_calc(
            lambda mu: jnp.broadcast_to(mu[None, :, :], (n_units, K, n_params)),
            mu=mu_k,
            name="beta_loc"
        )

    def make_beta_mixture(pvec, locs, scale_trils):
        return tfd.MixtureSameFamily(
            mixture_distribution=tfd.Categorical(probs=pvec),
            components_distribution=tfd.MultivariateNormalTriL(
                loc=locs,
                scale_tril=scale_trils[None]  # broadcast K over n_units
            )
        )

    beta_i = lsl.Var.new_param(
        value=jnp.zeros((n_units, n_params)),
        distribution=lsl.Dist(
            make_beta_mixture,
            pvec=pvec,
            locs=beta_loc,
            scale_trils=sigma_chol_k
        ),
        name="beta_i"
    )

    X_var   = lsl.Var.new_obs(data_dict["X"],        name="X_obs")
    idx_var = lsl.Var.new_obs(data_dict["unit_idx"], name="idx_obs")

    beta_expanded = lsl.Var.new_calc(
        lambda b, idx: b[idx], b=beta_i, idx=idx_var, name="beta_expanded"
    )
    logits = lsl.Var.new_calc(
        lambda x, b: jnp.einsum("nij,nj->ni", x, b),
        x=X_var, b=beta_expanded,
        name="logits"
    )
    y_var = lsl.Var.new_obs(
        data_dict["y"],
        distribution=lsl.Dist(tfd.Categorical, logits=logits),
        name="y"
    )

    return lsl.Model([y_var])

In [ ]:
# %% [markdown]
# ## 3. MCMC Inference Pipeline

# %%
def run_mixture_inference(model, data_dict, chains=4, warmup=1000, posterior=2000, seed=123):
    has_Z = data_dict.get("Z") is not None
    eb = gs.EngineBuilder(seed=seed, num_chains=chains)
    eb.set_model(gs.LieselInterface(model))
    eb.set_initial_values(model.state)
    
    # --- FIX: Split parameters into independent NUTS kernels ---
    # This prevents scaling conflicts during mass matrix adaptation
    eb.add_kernel(gs.NUTSKernel(["pvec_latent"]))
    eb.add_kernel(gs.NUTSKernel(["mu_k"]))
    eb.add_kernel(gs.NUTSKernel(["sigma_inv_chol_k_latent"]))
    
    if has_Z:
        eb.add_kernel(gs.NUTSKernel(["Delta"]))

    eb.add_kernel(gs.NUTSKernel(["beta_i"]))
    eb.set_duration(warmup_duration=warmup, posterior_duration=posterior)

    print("Starting NUTS Sampling — Mixture of Normals HMNL...")
    engine = eb.build()
    engine.sample_all_epochs()
    return engine.get_results(), engine.get_results().get_posterior_samples()

# Build and Run
hmnl_model = build_mixture_hmnl_model(
    choice_data,
    A_delta=1/16,    # Matched from R
    a_mu=0.01,       # Standard bayesm
    dirichlet_a=5.0  # Standard bayesm
)

start_time = time.time()
mcmc_results, posterior_samples = run_mixture_inference(
    hmnl_model, choice_data, chains=1, warmup=1000, posterior=5000
)
formatted_time = str(datetime.timedelta(seconds=int(time.time() - start_time)))
print(f"Sampling finished in {formatted_time}")

c:\Users\ThinkPad\Desktop\Repositories\BDCM\liesel_project\.venv\Lib\site-packages\jax\_src\numpy\array_methods.py:122: UserWarning: Explicitly requested dtype float64 requested in astype is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return lax_numpy.astype(self, dtype, copy=copy, device=device)
liesel.goose.builder - WARNING - No jitter functions provided. The initial values won't be jittered
liesel.goose.engine - INFO - Initializing kernels...


Starting NUTS Sampling — Mixture of Normals HMNL...


liesel.goose.engine - INFO - Done
liesel.goose.engine - INFO - Starting epoch: FAST_ADAPTATION, 75 transitions, 25 jitted together
100%|██████████████████████████████████████████| 3/3 [00:40<00:00, 13.38s/chunk]
liesel.goose.engine - WARNING - Errors per chain for kernel_01: 69 / 75 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_02: 7 / 75 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_03: 5 / 75 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_04: 2 / 75 transitions
liesel.goose.engine - INFO - Finished epoch
liesel.goose.engine - INFO - Starting epoch: SLOW_ADAPTATION, 25 transitions, 25 jitted together
100%|█████████████████████████████████████████| 1/1 [00:00<00:00, 474.79chunk/s]
liesel.goose.engine - WARNING - Errors per chain for kernel_00: 1 / 25 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_01: 25 / 25 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_02: 2 

In [ ]:
# %% [markdown]
# ## 4. Label Switching Resolution (PRA)

# %%
def pra(xi_star, xi_draws):
    m, K, J = xi_draws.shape
    perms     = np.empty((m, K), dtype=int)
    relabeled = np.empty_like(xi_draws)
    for t in range(m):
        cost = np.sum(
            (xi_star[:, None, :] - xi_draws[t, None, :, :]) ** 2, axis=-1
        )
        _, perms[t] = linear_sum_assignment(cost)
        relabeled[t] = xi_draws[t, perms[t], :]
    return perms, relabeled

def apply_permutations(draws, perms):
    m, K = perms.shape
    relabeled = np.empty_like(draws)
    for t in range(m):
        relabeled[t] = draws[t, perms[t]]
    return relabeled

def resolve_label_switching(samples_dict, pivot_chain_idx=0):
    print("\nStarting Pivotal Reordering Algorithm (PRA)...")
    mu             = samples_dict["mu_k"]
    n_chains, n_draws, K, n_params = mu.shape
    m              = n_chains * n_draws

    mu_flat        = mu.reshape(m, K, n_params)
    xi_star        = np.mean(mu[pivot_chain_idx], axis=0)  

    perms, relabeled_mu_flat = pra(xi_star, mu_flat)

    sorted_samples         = samples_dict.copy()
    sorted_samples["mu_k"] = relabeled_mu_flat.reshape(n_chains, n_draws, K, n_params)

    sigma       = samples_dict["sigma_inv_chol_k_latent"]
    sigma_shape = sigma.shape
    sigma_flat  = sigma.reshape(m, K, *sigma_shape[3:])
    sorted_samples["sigma_inv_chol_k_latent"] = apply_permutations(
        sigma_flat, perms
    ).reshape(sigma_shape)

    pvec_latent  = samples_dict["pvec_latent"]
    bijector_pvec = tfb.SoftmaxCentered()
    pvec_simplex  = bijector_pvec.forward(pvec_latent)

    pvec_flat = pvec_simplex.reshape(m, K)
    sorted_samples["pvec"] = apply_permutations(pvec_flat, perms).reshape(n_chains, n_draws, K)

    print("Label switching resolved — all chains are now aligned.")
    return sorted_samples

posterior_samples_sorted = resolve_label_switching(posterior_samples, pivot_chain_idx=0)